# Subject uniqueness check across releases

This notebook scans the raw files in `data/`, ignores `.pkl` files, extracts `sub-*` subject ids from file paths, and checks whether any subject appears in more than one release.

In [1]:
from pathlib import Path
from itertools import combinations
import contextlib
import gc
import io
import pickle
import re
import sys
import warnings

from IPython.display import display
import numpy as np
import pandas as pd
from tqdm import tqdm

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)


In [2]:
# Prefer ../data when the kernel starts in neurosned/notebooks/.
# Other candidates make the notebook robust when Jupyter starts elsewhere.
DATA_DIR_CANDIDATES = [
    Path("../data"),
    Path("data"),
    Path("ayana_experiments/neurosned/data"),
]

DATA_DIR = next((path.resolve() for path in DATA_DIR_CANDIDATES if path.exists()), None)
IGNORED_SUFFIXES = {".pkl"}

if DATA_DIR is None:
    candidates = ", ".join(str(path) for path in DATA_DIR_CANDIDATES)
    raise FileNotFoundError(f"Could not find data directory. Tried: {candidates}")

DATA_DIR


PosixPath('/home/qdata/ayana_experiments/neurosned/data')

In [3]:
def extract_subject_id(path_parts, file_name):
    """Find a BIDS-style subject id like sub-NDAR... in path components or filename."""
    for part in path_parts:
        if part.startswith("sub-"):
            return part

    match = re.search(r"sub-[A-Za-z0-9]+", file_name)
    return match.group(0) if match else None


records = []
skipped_files = []

for path in sorted(DATA_DIR.rglob("*")):
    if not path.is_file():
        continue
    if path.suffix.lower() in IGNORED_SUFFIXES:
        continue

    relative_path = path.relative_to(DATA_DIR)
    path_parts = relative_path.parts
    release = path_parts[0] if path_parts else None
    subject = extract_subject_id(path_parts, path.name)

    if release is None or subject is None:
        skipped_files.append(str(relative_path))
        continue

    records.append(
        {
            "release": release,
            "subject": subject,
            "suffix": path.suffix.lower(),
            "path": str(relative_path),
        }
    )

files_df = pd.DataFrame(records)

print(f"Scanned non-pkl files: {len(files_df):,}")
print(f"Skipped non-pkl files without release/subject id: {len(skipped_files):,}")

if skipped_files:
    display(pd.DataFrame({"skipped_path": skipped_files}).head(50))

files_df.head()

Scanned non-pkl files: 20,108
Skipped non-pkl files without release/subject id: 0


,release,subject,suffix,path
0,EEG2025r1,sub-NDARAC904DMU,.tsv,EEG2025r1/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-contrastChangeDetection_run-1_channels.tsv
1,EEG2025r1,sub-NDARAC904DMU,.bdf,EEG2025r1/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-contrastChangeDetection_run-1_eeg.bdf
2,EEG2025r1,sub-NDARAC904DMU,.json,EEG2025r1/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-contrastChangeDetection_run-1_eeg.json
3,EEG2025r1,sub-NDARAC904DMU,.tsv,EEG2025r1/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-contrastChangeDetection_run-1_events.tsv
4,EEG2025r1,sub-NDARAC904DMU,.tsv,EEG2025r1/sub-NDARAC904DMU/eeg/sub-NDARAC904DMU_task-contrastChangeDetection_run-2_channels.tsv


In [4]:
release_summary = (
    files_df.groupby("release")
    .agg(
        n_subjects=("subject", "nunique"),
        n_files=("path", "size"),
        file_types=("suffix", lambda values: ", ".join(sorted(set(values)))),
    )
    .sort_index()
)

release_summary

,n_subjects,n_files,file_types
release,,,
EEG2025r1,102,1172,".bdf, .json, .tsv"
EEG2025r10,79,848,".bdf, .json, .tsv"
EEG2025r11,293,2996,".bdf, .json, .tsv"
EEG2025r2,107,1204,".bdf, .json, .tsv"
EEG2025r3,138,1552,".bdf, .json, .tsv"
EEG2025r4,258,3000,".bdf, .json, .tsv"
EEG2025r5,260,2980,".bdf, .json, .tsv"
EEG2025r6,95,928,".bdf, .json, .tsv"
EEG2025r7,168,1672,".bdf, .json, .tsv"


In [5]:
subject_release_df = files_df[["subject", "release"]].drop_duplicates()

subject_release_counts = subject_release_df.groupby("subject")["release"].nunique()
repeated_subjects = subject_release_counts[subject_release_counts > 1].sort_index()

print(f"Total unique subjects: {subject_release_counts.size:,}")
print(f"Subjects present in more than one release: {len(repeated_subjects):,}")

if repeated_subjects.empty:
    print("OK: no subject is repeated across releases.")
else:
    repeated_detail = (
        subject_release_df[subject_release_df["subject"].isin(repeated_subjects.index)]
        .sort_values(["subject", "release"])
        .reset_index(drop=True)
    )
    display(repeated_detail)

Total unique subjects: 1,828
Subjects present in more than one release: 0
OK: no subject is repeated across releases.


In [6]:
subjects_by_release = {
    release: set(group["subject"])
    for release, group in subject_release_df.groupby("release")
}

overlap_rows = []
for release_a, release_b in combinations(sorted(subjects_by_release), 2):
    overlap = sorted(subjects_by_release[release_a] & subjects_by_release[release_b])
    overlap_rows.append(
        {
            "release_a": release_a,
            "release_b": release_b,
            "n_overlap_subjects": len(overlap),
            "overlap_subjects": ", ".join(overlap),
        }
    )

overlap_df = pd.DataFrame(overlap_rows)
overlap_df.sort_values(["n_overlap_subjects", "release_a", "release_b"], ascending=[False, True, True])

,release_a,release_b,n_overlap_subjects,overlap_subjects
0,EEG2025r1,EEG2025r10,0,
1,EEG2025r1,EEG2025r11,0,
2,EEG2025r1,EEG2025r2,0,
3,EEG2025r1,EEG2025r3,0,
4,EEG2025r1,EEG2025r4,0,
5,EEG2025r1,EEG2025r5,0,
6,EEG2025r1,EEG2025r6,0,
7,EEG2025r1,EEG2025r7,0,
8,EEG2025r1,EEG2025r8,0,
9,EEG2025r1,EEG2025r9,0,


In [7]:
# Compact view: only pairs with repeated subjects.
nonzero_overlap_df = overlap_df[overlap_df["n_overlap_subjects"] > 0].reset_index(drop=True)

if nonzero_overlap_df.empty:
    print("No pairwise release overlaps found.")
else:
    display(nonzero_overlap_df)

No pairwise release overlaps found.


## New validation split: R1-R8 train, R9-R10 validation, R11 holdout

This section follows the Challenge 1 data-preparation logic: load releases with `EEGChallengeDataset`, annotate trials, add stimulus anchors, remove anchors that are too late for the requested window, create event windows, inject metadata columns, and save the resulting `BaseConcatDataset` objects as `.pkl` files.

Requested split:
- `R1`-`R8` -> train
- `R9`-`R10` -> validation
- `R11` -> holdout test


In [8]:
NEW_VALIDATION_DIR = DATA_DIR / "new_validation"
NEW_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_CONFIG = [
    {
        "name": "r1_r8_train",
        "releases": [f"R{idx}" for idx in range(1, 9)],
        "outputs": {
            "2sec": NEW_VALIDATION_DIR / "r1_r8_train.pkl",
            "5sec": NEW_VALIDATION_DIR / "r1_r8_train_5sec.pkl",
        },
    },
    {
        "name": "r9_r10_val",
        "releases": ["R9", "R10"],
        "outputs": {
            "2sec": NEW_VALIDATION_DIR / "r9_r10_val.pkl",
            "5sec": NEW_VALIDATION_DIR / "r9_r10_val_5sec.pkl",
        },
    },
    {
        "name": "r11_test",
        "releases": ["R11"],
        "outputs": {
            "2sec": NEW_VALIDATION_DIR / "r11_test.pkl",
        },
    },
]

planned_outputs = []
for split in SPLIT_CONFIG:
    for window_kind, path in split["outputs"].items():
        planned_outputs.append(
            {
                "split": split["name"],
                "releases": ", ".join(split["releases"]),
                "window_kind": window_kind,
                "output_path": str(path.relative_to(DATA_DIR)),
            }
        )

pd.DataFrame(planned_outputs)


,split,releases,window_kind,output_path
0,r1_r8_train,"R1, R2, R3, R4, R5, R6, R7, R8",2sec,new_validation/r1_r8_train.pkl
1,r1_r8_train,"R1, R2, R3, R4, R5, R6, R7, R8",5sec,new_validation/r1_r8_train_5sec.pkl
2,r9_r10_val,"R9, R10",2sec,new_validation/r9_r10_val.pkl
3,r9_r10_val,"R9, R10",5sec,new_validation/r9_r10_val_5sec.pkl
4,r11_test,R11,2sec,new_validation/r11_test.pkl


In [9]:
def release_to_cache_dir_name(release):
    match = re.fullmatch(r"R(\d+)", release.upper())
    if match:
        return f"EEG2025r{int(match.group(1))}"
    return release


split_subject_rows = []
subjects_by_new_split = {}

for split in SPLIT_CONFIG:
    release_dirs = {release_to_cache_dir_name(release) for release in split["releases"]}
    split_subjects = set(subject_release_df.loc[subject_release_df["release"].isin(release_dirs), "subject"])
    subjects_by_new_split[split["name"]] = split_subjects
    split_subject_rows.append(
        {
            "split": split["name"],
            "release_dirs": ", ".join(sorted(release_dirs)),
            "n_subjects_from_data_scan": len(split_subjects),
        }
    )

split_overlap_rows = []
for split_a, split_b in combinations(subjects_by_new_split, 2):
    overlap = sorted(subjects_by_new_split[split_a] & subjects_by_new_split[split_b])
    split_overlap_rows.append(
        {
            "split_a": split_a,
            "split_b": split_b,
            "n_overlap_subjects": len(overlap),
            "overlap_subjects": ", ".join(overlap),
        }
    )

print("Subjects by requested split, based on files already present in data/:")
display(pd.DataFrame(split_subject_rows))

new_split_overlap_df = pd.DataFrame(split_overlap_rows)
print("Subject overlap between requested train/validation/test splits:")
display(new_split_overlap_df)


Subjects by requested split, based on files already present in data/:


,split,release_dirs,n_subjects_from_data_scan
0,r1_r8_train,"EEG2025r1, EEG2025r2, EEG2025r3, EEG2025r4, EEG2025r5, EEG2025r6, EEG2025r7, EEG2025r8",1227
1,r9_r10_val,"EEG2025r10, EEG2025r9",308
2,r11_test,EEG2025r11,293


Subject overlap between requested train/validation/test splits:


,split_a,split_b,n_overlap_subjects,overlap_subjects
0,r1_r8_train,r9_r10_val,0,
1,r1_r8_train,r11_test,0,
2,r9_r10_val,r11_test,0,


In [10]:
from eegdash.dataset import EEGChallengeDataset
from braindecode.datasets.base import BaseConcatDataset
from braindecode.preprocessing import preprocess, Preprocessor, create_windows_from_events
from eegdash.hbn.windows import (
    annotate_trials_with_target,
    add_aux_anchors,
    add_extras_columns,
    keep_only_recordings_with,
)

warnings.filterwarnings(
    "ignore", message="Omitted .* annotation.*outside data range", category=RuntimeWarning
)

ANCHOR = "stimulus_anchor"
SFREQ = 100
WINDOW_METADATA_KEYS = (
    "target",
    "rt_from_stimulus",
    "rt_from_trialstart",
    "stimulus_onset",
    "response_onset",
    "correct",
    "response_type",
)


def prepare_full_dataset(data_dir, release_list):
    all_datasets_list = []
    buf = io.StringIO()

    for release in tqdm(release_list, file=sys.stdout):
        with contextlib.redirect_stderr(buf):
            ds = EEGChallengeDataset(
                release=release,
                task="contrastChangeDetection",
                mini=False,
                description_fields=[
                    "subject",
                    "session",
                    "run",
                    "task",
                    "age",
                    "gender",
                    "sex",
                    "p_factor",
                ],
                cache_dir=data_dir,
            )
        all_datasets_list.append(ds)

    return BaseConcatDataset(all_datasets_list)


/home/qdata/miniconda3/envs/ayana/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def _recording_label(ds):
    desc = getattr(ds, "description", None)
    if desc is None:
        return str(getattr(ds, "filecache", "<unknown>"))

    parts = []
    for key in ("subject", "session", "run", "task"):
        if key in desc and pd.notna(desc[key]):
            parts.append(f"{key}={desc[key]}")
    return ", ".join(parts) or str(getattr(ds, "filecache", "<unknown>"))


def _bad_cached_bdf_reason(ds):
    path = getattr(ds, "filecache", None)
    if path is None:
        return None

    path = Path(path)
    if path.suffix.lower() != ".bdf" or not path.exists():
        return None

    size = path.stat().st_size
    if size < 256:
        return f"cached BDF is too small ({size} bytes)"

    with path.open("rb") as f:
        header = f.read(256)

    if not header.strip(bytes([0])):
        return "first 256 BDF header bytes are all null"

    header_size = header[184:192].decode("latin-1", errors="ignore").strip()
    if not header_size.isdigit():
        return f"invalid BDF header-size field: {header_size!r}"

    if size < int(header_size):
        return f"cached BDF is smaller than its header ({size} < {header_size})"

    return None


def drop_bad_cached_bdf_headers(concat_ds):
    kept = []
    skipped = []

    for ds in tqdm(concat_ds.datasets, desc="Checking cached BDF headers", file=sys.stdout):
        reason = _bad_cached_bdf_reason(ds)
        if reason is None:
            kept.append(ds)
        else:
            skipped.append(
                {
                    "recording": _recording_label(ds),
                    "path": str(getattr(ds, "filecache", "")),
                    "reason": reason,
                }
            )

    skipped_df = pd.DataFrame(skipped)
    if skipped_df.empty:
        print("No invalid cached BDF headers found.")
    else:
        print(f"Dropping {len(skipped_df)} recording(s) with invalid cached BDF headers.")
        display(skipped_df.head(20))
        if len(skipped_df) > 20:
            print(f"... {len(skipped_df) - 20} more skipped recording(s) not shown")

    if not kept:
        raise RuntimeError("All recordings were dropped by the cached BDF header check.")

    return BaseConcatDataset(kept, target_transform=getattr(concat_ds, "target_transform", None)), skipped_df


def _is_skippable_recording_error(exc):
    msg = str(exc).lower()
    skippable_fragments = (
        "bad bdf",
        "bad edf",
        "data file unreadable",
        "error reading",
        "file not found",
        "no such file",
        "could not load raw data",
    )
    return any(fragment in msg for fragment in skippable_fragments)


def preprocess_skip_bad(concat_ds, preprocessors):
    kept = []
    skipped = []

    for ds in tqdm(concat_ds.datasets, desc="Preprocessing recordings", file=sys.stdout):
        try:
            single_ds = BaseConcatDataset([ds])
            preprocess(single_ds, preprocessors, n_jobs=1)
            kept.append(single_ds.datasets[0])
        except Exception as exc:
            if not _is_skippable_recording_error(exc):
                raise
            if hasattr(ds, "_raw"):
                ds._raw = None
            skipped.append(
                {
                    "recording": _recording_label(ds),
                    "path": str(getattr(ds, "filecache", "")),
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    skipped_df = pd.DataFrame(skipped)
    if skipped_df.empty:
        print("Preprocessed all recordings successfully.")
    else:
        print(f"Skipped {len(skipped_df)} recording(s) during preprocessing.")
        display(skipped_df.head(20))
        if len(skipped_df) > 20:
            print(f"... {len(skipped_df) - 20} more skipped recording(s) not shown")

    if not kept:
        raise RuntimeError("All recordings failed during preprocessing.")

    return BaseConcatDataset(kept, target_transform=getattr(concat_ds, "target_transform", None)), skipped_df


def build_preprocessed_release_dataset(data_dir, release_list):
    print(f"Loading releases: {release_list}")
    dataset_ccd = prepare_full_dataset(data_dir=data_dir, release_list=release_list)
    print(f"Loaded recordings: {len(dataset_ccd.datasets):,}")

    transformation_offline = [
        Preprocessor(
            annotate_trials_with_target,
            target_field="rt_from_stimulus",
            epoch_length=2.0,
            require_stimulus=True,
            require_response=True,
            apply_on_array=False,
        ),
        Preprocessor(add_aux_anchors, apply_on_array=False),
    ]

    dataset_ccd, skipped_bad_bdf_headers = drop_bad_cached_bdf_headers(dataset_ccd)
    dataset_ccd, skipped_preprocessing = preprocess_skip_bad(dataset_ccd, transformation_offline)

    return dataset_ccd, skipped_bad_bdf_headers, skipped_preprocessing


In [12]:
def remove_late_anchors(dataset, anchor=ANCHOR, shift=0.5, winlen=2.0):
    log = []
    for idx, bd in enumerate(dataset.datasets):
        raw = bd.raw
        ann = raw.annotations
        recording_len_s = float(raw.times[-1])

        desc = np.asarray(ann.description, dtype=str)
        onset = np.asarray(ann.onset, dtype=float)

        is_anchor = desc == anchor
        n_before = int(is_anchor.sum())
        too_late = is_anchor & ((onset + shift + winlen) > recording_len_s + 1e-9)
        n_removed = int(too_late.sum())

        if n_removed > 0:
            log.append(
                {
                    "idx": idx,
                    "subject": bd.description.get("subject", "NA"),
                    "run": bd.description.get("run", "NA"),
                    "release": bd.description.get("release_number", "NA"),
                    "n_anchors_before": n_before,
                    "n_removed": n_removed,
                    "max_onset_removed": float(onset[too_late].max()),
                    "rec_len_s": recording_len_s,
                }
            )

        raw.set_annotations(ann[np.where(~too_late)[0]], verbose=False)

    if log:
        df_log = pd.DataFrame(log).sort_values(["n_removed", "n_anchors_before"], ascending=False)
        print(f"Removed anchors in total: {int(df_log['n_removed'].sum())}")
        print(f"Records with removed anchors: {len(df_log)}")
        display(df_log.head(20))

    return keep_only_recordings_with(anchor, dataset)


def create_release_windows(preprocessed_dataset, shift_after_stim, window_len_s, epoch_len_s):
    dataset_with_anchors = keep_only_recordings_with(ANCHOR, preprocessed_dataset)
    dataset_with_anchors = remove_late_anchors(
        dataset_with_anchors,
        anchor=ANCHOR,
        shift=shift_after_stim,
        winlen=window_len_s,
    )

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        windows = create_windows_from_events(
            dataset_with_anchors,
            mapping={ANCHOR: 0},
            trial_start_offset_samples=int(shift_after_stim * SFREQ),
            trial_stop_offset_samples=int((shift_after_stim + window_len_s) * SFREQ),
            window_size_samples=int(epoch_len_s * SFREQ),
            window_stride_samples=SFREQ,
            preload=True,
        )

    windows = add_extras_columns(
        windows,
        dataset_with_anchors,
        desc=ANCHOR,
        keys=WINDOW_METADATA_KEYS,
    )
    return windows


def save_pickle_dataset(dataset, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("wb") as f:
        pickle.dump(dataset, f)
    print(f"Saved {output_path.relative_to(DATA_DIR)} ({len(dataset):,} windows)")


def summarize_windows(dataset, split_name, window_kind):
    metadata = dataset.get_metadata()
    return {
        "split": split_name,
        "window_kind": window_kind,
        "n_windows": len(dataset),
        "n_subjects": metadata["subject"].nunique() if "subject" in metadata else None,
        "target_min": metadata["target"].min() if "target" in metadata else None,
        "target_max": metadata["target"].max() if "target" in metadata else None,
    }


In [13]:
BUILD_NEW_VALIDATION_PICKLES = True

if BUILD_NEW_VALIDATION_PICKLES:
    build_summary_rows = []
    skipped_summary_rows = []

    for split in SPLIT_CONFIG:
        print("=" * 100)
        print(f"Building {split['name']} from releases: {split['releases']}")

        preprocessed_dataset, skipped_bad_bdf_headers, skipped_preprocessing = build_preprocessed_release_dataset(
            DATA_DIR,
            split["releases"],
        )

        skipped_summary_rows.append(
            {
                "split": split["name"],
                "skipped_bad_bdf_headers": len(skipped_bad_bdf_headers),
                "skipped_preprocessing": len(skipped_preprocessing),
            }
        )

        windows_2sec = create_release_windows(
            preprocessed_dataset,
            shift_after_stim=0.5,
            window_len_s=2.0,
            epoch_len_s=2.0,
        )
        save_pickle_dataset(windows_2sec, split["outputs"]["2sec"])
        build_summary_rows.append(summarize_windows(windows_2sec, split["name"], "2sec"))
        del windows_2sec
        gc.collect()

        if "5sec" in split["outputs"]:
            windows_5sec = create_release_windows(
                preprocessed_dataset,
                shift_after_stim=0.0,
                window_len_s=5.0,
                epoch_len_s=5.0,
            )
            save_pickle_dataset(windows_5sec, split["outputs"]["5sec"])
            build_summary_rows.append(summarize_windows(windows_5sec, split["name"], "5sec"))
            del windows_5sec
            gc.collect()

        del preprocessed_dataset
        gc.collect()

    build_summary = pd.DataFrame(build_summary_rows)
    skipped_summary = pd.DataFrame(skipped_summary_rows)

    print("Done. Created new validation pickles in:")
    print(NEW_VALIDATION_DIR)
    display(build_summary)
    display(skipped_summary)
else:
    print("BUILD_NEW_VALIDATION_PICKLES is False, so no files were created.")


Building r1_r8_train from releases: ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']
Loading releases: ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']
  0%|          | 0/8 [00:00<?, ?it/s]

╭─────────────────────────────────────── EEG 2025 Competition Data Notice ────────────────────────────────────────╮
│ This object loads the HBN dataset that has been preprocessed for the EEG Challenge:                             │
│   * Downsampled from 500Hz to 100Hz                                                                             │
│   * Bandpass filtered (0.5-50 Hz)                                                                               │
│                                                                                                                 │
│ For full preprocessing applied for competition details, see:                                                    │
│   ]8;id=85858;https://github.com/eeg2025/downsample-datasets\https://github.com/eeg2025/downsample-datasets]8;;\                                                                │
│                                                                                                                 │
│ The HBN dataset have some preprocessing applied by the HBN team:                                                │
│   * Re-reference (Cz Channel)                                                                                   │
│                                                                                                                 │
│ IMPORTANT: The data accessed via `EEGChallengeDataset` is NOT identical to what you get from ]8;id=85859;https://github.com/eegdash/EEGDash/blob/develop/eegdash/api.py\EEGDashDataset]8;;\     │
│ directly.                                                                                                       │
│ If you are participating in the competition, always use `EEGChallengeDataset` to ensure consistency with the    │
│ challenge data.                                                                                                 │
╰────────────────────────────────────────── Source: EEGChallengeDataset ──────────────────────────────────────────╯

[06/09/26 12:45:46] INFO     Auto-corrected misrouted storage.base for dataset EEG2025r1: None ->    ]8;id=85866;file:///home/qdata/miniconda3/envs/ayana/lib/python3.12/site-packages/eegdash/dataset/dataset.py\dataset.py]8;;\:]8;id=85867;file:///home/qdata/miniconda3/envs/ayana/lib/python3.12/site-packages/eegdash/dataset/dataset.py#413\413]8;;\
                             s3://nemar/EEG2025r1                                                                  

 12%|█▎        | 1/8 [00:02<00:15,  2.24s/it]

╭─────────────────────────────────────── EEG 2025 Competition Data Notice ────────────────────────────────────────╮
│ This object loads the HBN dataset that has been preprocessed for the EEG Challenge:                             │
│   * Downsampled from 500Hz to 100Hz                                                                             │
│   * Bandpass filtered (0.5-50 Hz)                                                                               │
│                                                                                                                 │
│ For full preprocessing applied for competition details, see:                                                    │
│   ]8;id=85870;https://github.com/eeg2025/downsample-datasets\https://github.com/eeg2025/downsample-datasets]8;;\                                                                │
│                                                                                                                 │
│ The HBN dataset have some preprocessing applied by the HBN team:                                                │
│   * Re-reference (Cz Channel)                                                                                   │
│                                                                                                                 │
│ IMPORTANT: The data accessed via `EEGChallengeDataset` is NOT identical to what you get from ]8;id=85871;https://github.com/eegdash/EEGDash/blob/develop/eegdash/api.py\EEGDashDataset]8;;\     │
│ directly.                                                                                                       │
│ If you are participating in the competition, always use `EEGChallengeDataset` to ensure consistency with the    │
│ challenge data.                                                                                                 │
╰────────────────────────────────────────── Source: EEGChallengeDataset ──────────────────────────────────────────╯

...

Saved new_validation/r11_test.pkl (15,751 windows)


/home/qdata/miniconda3/envs/ayana/lib/python3.12/site-packages/braindecode/datasets/base.py:1322: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_dfs)


Done. Created new validation pickles in:
/home/qdata/ayana_experiments/neurosned/data/new_validation


,split,window_kind,n_windows,n_subjects,target_min,target_max
0,r1_r8_train,2sec,75160,1228,0.0,2.498
1,r1_r8_train,5sec,75157,1228,0.0,2.498
2,r9_r10_val,2sec,17867,308,0.0,2.410
3,r9_r10_val,5sec,17867,308,0.0,2.410
4,r11_test,2sec,15751,292,0.0,2.420


,split,skipped_bad_bdf_headers,skipped_preprocessing
0,r1_r8_train,1,0
1,r9_r10_val,0,0
2,r11_test,0,0
